# THC qubitized phase estimation

This example composes canonical quantum phase estimation (QPE) with the controlled qubitization walk for the BLISS-THC Hamiltonian in [Caesura et al., arXiv:2501.06165](https://arxiv.org/abs/2501.06165). It uses 16 spatial orbitals and THC rank 16: a non-trivial instance containing 152 alias-sampled coefficients and 15 neighboring Givens rotations in each orbital transformation.

The implementation is adapted from the PREPARE and Select circuit construction in the original paper. Here, that construction is expressed using the repository's reusable alias-sampling, `SelectTHCCntrl`, controlled-LCU, qubitization, and QPE components.

The complete circuit is too large for practical statevector simulation. Its constituent algorithms are tested numerically on smaller systems, while this notebook serves as an integration example: it assembles the full workflow at a realistic size and checks that the resulting Guppy program compiles.

The construction is layered rather than monolithic:

```text
THC coefficients and rotations
        ↓
PREPARE + controlled SELECT + UNPREPARE
        ↓
LCUCntrl
        ↓
controlled qubitization = controlled LCU + controlled reflection
        ↓
QPE powered oracle
```

Each layer has a typed interface. The classical THC data can come from a chemistry workflow, and any signature-compatible PREPARE or SELECT can replace the generated oracle. After UNPREPARE, the cleared alias garbage is discarded and the standard controlled reflection acts only on the retained index register.

In [1]:
from typing import no_type_check

from guppylang import guppy
from guppylang.std.builtins import array, output
from guppylang.std.quantum import (
    collect_measurements,
    discard,
    discard_array,
    h,
    measure_array,
    qubit,
)

from guppyalgos.primitives.gate_decompositions.cnx import cnx
from guppyalgos.algorithms.block_encoding.lcu import LCUCntrl
from guppyalgos.algorithms.phase_estimation import qpe
from guppyalgos.primitives.subroutines.reflection import ReflectionCntrl
from guppyalgos.primitives.rotations import GivensCascadePhaseGradient
from guppyalgos.algorithms.state_preparation.alias_sampling import (
    AliasSamplingRegs,
    alias_samp_prep,
    discard_alias_sampling_garbage,
)
from guppyalgos.primitives.state_preparation.phase_gradient import Convention, phase_gradient
from guppyalgos.algorithms.block_encoding.thc import (
    SelectTHCCntrl,
    SelectTHCCntrlRegs,
    THCWalkTargetRegs,
    build_thc_lcu_data,
    generate_thc_parameters,
    load_select_registers,
)
from guppyalgos.utils import qarray, transversal

## 1. Supply classical THC data

`THCParameters` is the boundary between classical preprocessing and circuit construction. The coefficient arrays specify the one- and two-body weights; each row of the rotation arrays contains the neighboring-Givens angles for one orbital transformation. Angles are expressed as fractions of a full turn.

To place both kinds of coefficient in one alias-sampling table, picture the one-body terms as an extra column appended to the symmetric two-body matrix. The public term table represents entries in this column with `nu=None`; only the internal QROM encoding replaces `None` with the paper's reserved integer `nu=M`. A one-body term therefore still has only the meaningful index `mu`.

For this standalone example, `generate_thc_parameters` supplies deterministic non-zero data. A chemistry application would replace this one call with coefficients and rotations obtained from its classical factorization workflow. With 16 orbitals and rank 16, the flattened LCU contains 136 two-body coefficients and 16 one-body coefficients.

In [2]:
n_orbitals = 16
thc_rank = 16
parameters = generate_thc_parameters(n_orbitals, thc_rank, seed=26)

thc_lcu_data = build_thc_lcu_data(
    parameters,
    rotation_precision_bits=8,
    alias_precision_bits=12,
)

{
    "orbitals": n_orbitals,
    "THC rank": thc_rank,
    "LCU coefficients": len(thc_lcu_data.alias_probabilities),
    "Givens rotations per transform": thc_lcu_data.n_givens,
    "alias index qubits": thc_lcu_data.n_alias_qubits,
}

{'orbitals': 16,
 'THC rank': 16,
 'LCU coefficients': 152,
 'Givens rotations per transform': 15,
 'alias index qubits': 8}

## 2. Inspect the composable LCU oracles

`build_thc_lcu_data` is limited to classical preprocessing:

- It normalizes the THC coefficient magnitudes.
- It encodes the Select records and Givens angles as QROM tables.
- It returns the resulting Guppy QROMs and register dimensions.
- It does **not** choose a rotation method or construct PREPARE, SELECT, or UNPREPARE.

This notebook makes the quantum choices explicitly: alias-sampling PREPARE and a phase-gradient Givens cascade. Another compatible Select or rotator could use the same preprocessed data. The main returned values are:

- `alias_probabilities` contains the normalized coefficient magnitudes used to construct alias-sampling PREPARE below.
- `select_data_loader` is the precomputed QROM for the sampled THC indices and flags.
- `qrom_1_and_2_body` and `qrom_2_body` load the combined and two-body Givens angles.
- The standard `AliasSamplingRegs` and `SelectTHCCntrlRegs` types compose into the PREPARE register below.

Below, `alias_samp_prep` constructs PREPARE from the normalized magnitudes. The phase-gradient preparation and cascade are also selected here, outside the data builder.

In [ ]:
alias_prepare = alias_samp_prep(
    thc_lcu_data.alias_probabilities,
    thc_lcu_data.alias_precision,
)
select_data_qrom = thc_lcu_data.select_data_loader
qrom_1_and_2_body = thc_lcu_data.qrom_1_and_2_body
qrom_2_body = thc_lcu_data.qrom_2_body

n_alias_qubits = thc_lcu_data.n_alias_qubits
n_index_qubits = thc_lcu_data.n_index_qubits
n_keep_qubits = thc_lcu_data.n_keep_qubits
n_modes = thc_lcu_data.n_modes
n_givens = thc_lcu_data.n_givens
n_phase_qubits = thc_lcu_data.rotation_precision_bits

prepare_phase_gradient = phase_gradient(
    n_phase_qubits,
    convention=Convention.Standard,
)

@guppy.struct
class THCPrepareRegs:
    """Temporary LCU registers plus the phase gradient borrowed from QPE."""

    alias_sampling: AliasSamplingRegs[n_alias_qubits, n_keep_qubits]
    select: SelectTHCCntrlRegs[n_index_qubits]
    phase_gradient: array[qubit, n_phase_qubits]

@guppy
@no_type_check
def prepare(regs: THCPrepareRegs) -> None:
    """Prepare the alias distribution and load its THC Select record."""
    alias_prepare(
        regs.alias_sampling.index,
        regs.alias_sampling.alternative,
        regs.alias_sampling.keep,
        regs.alias_sampling.comparison,
        regs.alias_sampling.comparison_result,
        False,
    )
    load_select_registers(select_data_qrom, regs.alias_sampling.index, regs.select)

@guppy
@no_type_check
def cntrl_select(
    control: qubit,
    prep_regs: THCPrepareRegs,
    target_regs: THCWalkTargetRegs[n_modes],
) -> None:
    """Apply the THC Select using the prepared phase-gradient state."""
    cascade = GivensCascadePhaseGradient(prep_regs.phase_gradient)
    select = SelectTHCCntrl(qrom_1_and_2_body, qrom_2_body, cascade, cnx)
    select.compose(control, prep_regs.select, target_regs)
    prep_regs.phase_gradient = select.cascade.phase_gradient

@guppy
@no_type_check
def unprepare(regs: THCPrepareRegs) -> None:
    """Clear the THC Select record and reverse alias preparation."""
    load_select_registers(select_data_qrom, regs.alias_sampling.index, regs.select)
    alias_prepare(
        regs.alias_sampling.index,
        regs.alias_sampling.alternative,
        regs.alias_sampling.keep,
        regs.alias_sampling.comparison,
        regs.alias_sampling.comparison_result,
        True,
    )

@guppy.struct
class THCQPERegs:
    """Persistent quantum registers used by THC QPE."""

    alias_index: array[qubit, n_alias_qubits]
    target: THCWalkTargetRegs[n_modes]
    phase_gradient: array[qubit, n_phase_qubits]

## 3. Build the controlled block encoding

`LCUCntrl` manages the registers in this order:

- **Persistent QPE registers:** the prepare (alias) index, spin registers, and phase-gradient state enter in `THCQPERegs`.
- **Temporary workspace:** each call allocates the alias-sampling workspace and the THC Select indices and flags.
- **PREPARE:** the alias index is prepared and the selected THC record is loaded into the temporary Select registers.
- **SELECT:** the selected operation acts on the persistent spin registers and temporarily uses the phase-gradient state.
- **UNPREPARE:** the Select record and alias workspace are returned to $|0\rangle$.
- **Cleanup:** cleared workspace is discarded, while the alias index and phase gradient are moved back into `THCQPERegs`.

No persistent quantum state is copied. Only `THCQPERegs` and the borrowed control qubit survive the function.

In [ ]:
@guppy
@no_type_check
def cntrl_block_encoding(
    control: qubit,
    regs: THCQPERegs,
) -> None:
    """Apply the controlled THC block encoding."""
    # Move the persistent prepare index into temp alias-sampling workspace.
    alias_regs = AliasSamplingRegs(
        regs.alias_index,
        qarray(n_alias_qubits),
        qarray(n_keep_qubits),
        qarray(n_keep_qubits),
        qubit(),
    )
    # These Select registers are loaded during PREPARE and cleared by UNPREPARE.
    select_regs = SelectTHCCntrlRegs(
        qubit(),
        qubit(),
        qarray(n_index_qubits),
        qarray(n_index_qubits),
    )
    # The prepared phase gradient is borrowed from the persistent QPE state.
    prep_regs = THCPrepareRegs(
        alias_regs, select_regs, regs.phase_gradient
    )
    block_encoding = LCUCntrl(
        prepare,
        cntrl_select,
        unprepare,
    )
    block_encoding.compose(control, prep_regs, regs.target)

    # Keep the alias index, but consume the cleared alias-sampling workspace.
    regs.alias_index = discard_alias_sampling_garbage(
        prep_regs.alias_sampling.index,
        prep_regs.alias_sampling.alternative,
        prep_regs.alias_sampling.keep,
        prep_regs.alias_sampling.comparison,
        prep_regs.alias_sampling.comparison_result,
    )
    # UNPREPARE also cleared the temporary Select record.
    discard(prep_regs.select.one_body_flag)
    discard(prep_regs.select.coefficient_sign)
    discard_array(prep_regs.select.first_index_qreg)
    discard_array(prep_regs.select.second_index_qreg)
    # Move the reusable phase-gradient state back into the persistent QPE bundle.
    regs.phase_gradient = prep_regs.phase_gradient

## 4. Use the block encoding in qubitized phase estimation

Each controlled walk step has two operations:

1. `cntrl_block_encoding` applies the LCU and removes all temporary workspace.
2. `ReflectionCntrl` reflects the persistent alias index about $|0\rangle$.

QPE passes the same `THCQPERegs` bundle through every requested walk power. Its alias index, spin registers, and phase-gradient state therefore remain available for the next step.

In [6]:
@guppy
@no_type_check
def cntrl_walk_power(
    control: qubit,
    unitary_regs: THCQPERegs,
    power: int,
) -> None:
    """Apply the requested power of the controlled THC walk."""
    for _ in range(power):
        cntrl_block_encoding(control, unitary_regs)
        ReflectionCntrl(cnx[n_alias_qubits]).compose(
            control, unitary_regs.alias_index
        )

## 5. Compose with canonical QPE

A three-qubit phase register asks the powered oracle for walk powers 1, 2, and 4 before applying the inverse QFT. A useful energy estimate requires preparing an eigenstate, or a state with significant overlap with an eigenstate, in the spin registers. This example leaves them in the computational zero state because its purpose is to demonstrate and type-check the complete THC-QPE composition.

The resulting circuit is intentionally too large for a useful notebook statevector test. `check()` validates the Guppy composition and proves that every temporary alias register is consumed correctly.

In [7]:
n_qpe_qubits = 3

@guppy
@no_type_check
def main() -> None:
    phase_reg = qarray(n_qpe_qubits)
    alias_index = qarray(n_alias_qubits)
    target_regs = THCWalkTargetRegs(
        qarray(n_modes),
        qarray(n_modes),
    )
    phase_gradient_state = qarray(n_phase_qubits)
    prepare_phase_gradient(phase_gradient_state)
    unitary_regs = THCQPERegs(
        alias_index, target_regs, phase_gradient_state
    )
    transversal(h, phase_reg)

    qpe(phase_reg, unitary_regs, cntrl_walk_power)

    output("phase", collect_measurements(measure_array(phase_reg)))
    discard_array(unitary_regs.alias_index)
    discard_array(unitary_regs.target.spin_up)
    discard_array(unitary_regs.target.spin_down)
    discard_array(
        unitary_regs.phase_gradient
    )

main.check()
"THC qubitized phase estimation type checked successfully."

'Controlled THC walk type checked successfully.'